# Mfumo wa Kugundua Vimelea na Kupendekeza Dawa
### (Image Detection + Drug Recommendation Pipeline)

Notebook hii inaunganisha:
1. **Utambuzi wa picha (YOLOv8)** - kutoka `AMR.ipynb` - hugundua seli za damu / vimelea (malaria) au bakteria kwenye picha za darubini
2. **Data ya dawa (ChEMBL/PubChem)** - kutoka `API.py` - efficacy (IC50/EC50) na sasa pia toxicity (CC50/TC50)
3. **Injini ya kuunganisha (mapping + ranking)** - daraja jipya kati ya vimelea vilivyogundulika na dawa zinazofaa/hatari

---

## ⚠️ TAHADHARI YA KIMAADILI NA KISHERIA (SOMA KABLA YA KUENDELEA)

Hii ni **prototype ya utafiti/maendeleo (R&D)**. SI kifaa cha uchunguzi wa kimatibabu kilichoidhinishwa.

Kabla ya matumizi yoyote na wagonjwa halisi, mfumo huu unahitaji:

| # | Sharti |
|---|--------|
| 1 | **Uthibitisho wa kikliniki** - kupimwa dhidi ya ground truth halisi ya wagonjwa, siyo IC50 ya maabara pekee |
| 2 | **Ukaguzi wa mtaalamu** - daktari/pharmacologist aliyesajiliwa lazima apitie kila pendekezo |
| 3 | **Kibali cha mamlaka** - TMDA (Tanzania), FDA, EMA, au mamlaka husika ya nchi yako |
| 4 | **Human-in-the-loop** - matokeo ya mfumo ni PENDEKEZO tu, siyo uamuzi wa mwisho wa matibabu |

Data ya ChEMBL/PubChem ni ya **in-vitro (maabara)** - haithibitishi usalama/ufanisi kwa binadamu moja kwa moja.


## Sehemu 0: Usanidi (Setup)

Sakinisha maktaba zinazohitajika.

In [ ]:
import sys
!{sys.executable} -m pip install ultralytics chembl_webresource_client rdkit pandas requests -q
print("Maktaba zimesakinishwa.")

## Sehemu 1: Utambuzi wa Picha (YOLOv8) — kutoka `AMR.ipynb`

Hii ni muhtasari wa mchakato wa AMR.ipynb: kutrain YOLOv8 kugundua seli/vimelea kwenye picha.
Kama tayari una `best.pt` iliyofunzwa, unaweza kuruka moja kwa moja hadi **Sehemu 4** na kutumia njia yake.

Vinginevyo, fuata hatua hizi (zinahitaji Google Colab + Google Drive yenye dataset):

In [ ]:
# --- Hatua 1.1: Pandisha Google Drive (Colab pekee) ---
# from google.colab import drive
# drive.mount('/content/drive')

# --- Hatua 1.2: Eleza njia za dataset zako ---
import os

dataset1_path = '/content/drive/MyDrive/BCCD.v4-416x416_aug.yolov8'   # malaria/seli za damu
dataset2_path = '/content/drive/MyDrive/bacteria detection.v3i.yolov8'  # bakteria
combined_dataset_path = '/content/drive/MyDrive/jumla'

print(f"Dataset 1 (malaria/seli): {dataset1_path}")
print(f"Dataset 2 (bakteria): {dataset2_path}")
print(f"Dataset iliyounganishwa: {combined_dataset_path}")

In [ ]:
# --- Hatua 1.3: Train YOLOv8 (fanya hivi baada ya kuandaa data.yaml na kuunganisha data) ---
from ultralytics import YOLO

# model = YOLO('yolov8n.pt')
# results = model.train(
#     data=os.path.join(combined_dataset_path, 'data.yaml'),
#     epochs=50,
#     imgsz=640,
#     batch=16,
#     name='combined_dataset_yolov8_training'
# )
# best_model_path = results.save_dir / 'weights' / 'best.pt'
# print(f"Model bora imehifadhiwa: {best_model_path}")

print("Angalia AMR.ipynb kwa maelezo kamili ya hatua za kuandaa dataset na kutrain.")

## Sehemu 2: Ukusanyaji wa Data za Dawa (ChEMBL/PubChem) — kutoka `API.py`

Hii inavuta:
- **Efficacy data** (IC50/EC50) — je molekuli inafanya kazi dhidi ya target?
- **NB:** Sehemu hii ilikuwa na efficacy pekee; toxicity imeongezwa kwenye Sehemu 3.

In [ ]:
import pandas as pd
import requests
from chembl_webresource_client.new_client import new_client
from rdkit import Chem
import time

print("--- Kupakua Data za Efficacy kutoka ChEMBL ---")

activity = new_client.activity
chembl_results = activity.filter(standard_type__in=["IC50", "EC50"]).filter(relation="=")

raw_data = []
SAMPLE_SIZE = 5000  # ongeza kama unahitaji dataset kubwa zaidi

for item in chembl_results[:SAMPLE_SIZE]:
    smiles = item.get('canonical_smiles')
    val = item.get('standard_value')
    units = item.get('standard_units')
    target = item.get('target_chembl_id')

    if smiles and val:
        is_effective = 1 if float(val) <= 1000 else 0
        raw_data.append({
            'SMILES': smiles,
            'Target_ID': target,
            'Value_nM': float(val),
            'Units': units,
            'Is_Effective': is_effective,
            'Source': 'ChEMBL'
        })

df_chembl = pd.DataFrame(raw_data)
print(f"ChEMBL: rekodi {len(df_chembl)} zimepatikana.")

In [ ]:
def validate_smiles(smiles):
    try:
        m = Chem.MolFromSmiles(smiles)
        return m is not None
    except Exception:
        return False

df_chembl['is_valid'] = df_chembl['SMILES'].apply(validate_smiles)
full_dataset = df_chembl[df_chembl['is_valid'] == True].drop(columns=['is_valid'])

output_file = "offline_drug_training_data.csv"
full_dataset.to_csv(output_file, index=False)
print(f"Dataset imehifadhiwa: {output_file} ({len(full_dataset)} molekuli)")

## Sehemu 3: Injini ya Kuunganisha — `mapping_engine.py`

Hii ndiyo **daraja lililokuwa likikosekana**: kutoka class ya YOLO (mf. "Schizont") kwenda `target_chembl_id` za ChEMBL.

Tunatumia jina la kisayansi la kiumbe (siyo ID fixed) ili kuepuka makosa ya uhusiano usio sahihi.

In [ ]:
CLASS_TO_ORGANISM = {
    # -------- Malaria (kutoka AMR.ipynb / BCCD dataset) --------
    "Ring": "Plasmodium falciparum",
    "Trophozoite": "Plasmodium falciparum",
    "Schizont": "Plasmodium falciparum",
    "Gametocyte": "Plasmodium falciparum",

    # -------- Bakteria (AMR halisi) --------
    # MIFANO TU. Badilisha kulingana na class_names za data.yaml yako halisi.
    "Gram_positive_cocci": "Staphylococcus aureus",
    "Gram_negative_rod": "Escherichia coli",
    "Mycobacterium": "Mycobacterium tuberculosis",
}

_target_cache = {}

def get_targets_for_organism(organism_name, limit=20):
    """Tafuta target_chembl_id zote za kiumbe fulani (organism)."""
    if organism_name in _target_cache:
        return _target_cache[organism_name]

    target = new_client.target
    results = target.filter(organism__icontains=organism_name).only(
        ["target_chembl_id", "pref_name", "target_type", "organism"]
    )[:limit]

    targets = list(results)
    _target_cache[organism_name] = targets
    return targets


def resolve_targets_for_class(yolo_class_name):
    """Class ya YOLO -> orodha ya target_chembl_id husika."""
    organism = CLASS_TO_ORGANISM.get(yolo_class_name)
    if organism is None:
        return {"error": f"Class '{yolo_class_name}' haipo kwenye mapping. Iongeze kwanza."}

    targets = get_targets_for_organism(organism)
    return {
        "yolo_class": yolo_class_name,
        "organism": organism,
        "target_chembl_ids": [t["target_chembl_id"] for t in targets],
        "target_details": targets,
    }

print("mapping_engine imepakiwa.")

## Sehemu 4: Injini ya Uamuzi — `decision_engine.py`

Hapa ndipo tunapoongeza **toxicity data** (iliyokuwa haipo kabisa) na kuhesabu **Selectivity Index**
(uwiano wa usalama dhidi ya nguvu ya dawa) ili kujua "bora" dhidi ya "madhara".

In [ ]:
def fetch_toxicity_data(target_chembl_ids=None, limit=2000):
    """Vuta CC50/TC50 (cytotoxicity dhidi ya seli za binadamu)."""
    activity = new_client.activity
    query = activity.filter(standard_type__in=["CC50", "TC50"], relation="=")

    rows = []
    for item in query[:limit]:
        smiles = item.get("canonical_smiles")
        val = item.get("standard_value")
        assay_desc = item.get("assay_description") or ""
        if smiles and val:
            rows.append({"SMILES": smiles, "Toxicity_nM": float(val), "Assay_Info": assay_desc})
    return pd.DataFrame(rows)


def rank_drug_candidates(efficacy_df, toxicity_df, target_chembl_ids, patient_allergy_smiles=None):
    """Panga dawa: bora zaidi -> madhara, kutokana na Selectivity Index."""
    candidates = efficacy_df[
        efficacy_df["Target_ID"].isin(target_chembl_ids) & (efficacy_df["Is_Effective"] == 1)
    ].copy()

    if candidates.empty:
        return pd.DataFrame(), "Hakuna dawa zenye ushahidi wa ufanisi dhidi ya target hizi."

    merged = candidates.merge(toxicity_df, on="SMILES", how="left")
    merged["Selectivity_Index"] = merged["Toxicity_nM"] / merged["Value_nM"]

    if patient_allergy_smiles:
        merged["is_allergy_risk"] = merged["SMILES"].isin(patient_allergy_smiles)
    else:
        merged["is_allergy_risk"] = False

    def classify(row):
        if row["is_allergy_risk"]:
            return "MADHARA - Epuka (mzio wa mgonjwa)"
        if pd.isna(row["Selectivity_Index"]):
            return "Efficacy ipo, toxicity haijulikani - tahadhari"
        if row["Selectivity_Index"] >= 10:
            return "Inapendekezwa - Bora"
        if row["Selectivity_Index"] >= 2:
            return "Inafaa - Mzuri wa kati"
        return "MADHARA - Selectivity Index ndogo (hatari ya sumu)"

    merged["recommendation"] = merged.apply(classify, axis=1)
    merged = merged.sort_values(by=["is_allergy_risk", "Selectivity_Index"], ascending=[True, False])

    cols = ["SMILES", "Target_ID", "Value_nM", "Toxicity_nM", "Selectivity_Index", "recommendation"]
    return merged[cols], "OK"

print("decision_engine imepakiwa.")

## Sehemu 5: Pipeline Kamili — Picha → Utambuzi → Pendekezo la Dawa

Kazi hii inaunganisha Sehemu 1-4 kuwa mtiririko mmoja: toa picha, pata pendekezo la dawa.

In [ ]:
from ultralytics import YOLO

def detect_classes(image_path, model_path, conf_threshold=0.5):
    """Tumia YOLO iliyofunzwa (best.pt) kugundua class kwenye picha."""
    model = YOLO(model_path)
    results = model.predict(source=image_path, conf=conf_threshold)

    detected_classes = set()
    for r in results:
        for box in r.boxes:
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            detected_classes.add(cls_name)
    return list(detected_classes)


def run_pipeline(image_path, model_path, efficacy_csv, patient_allergy_smiles=None):
    print(f"[1/4] Inasoma picha: {image_path}")
    detected = detect_classes(image_path, model_path)
    print(f"      Vimegunduliwa: {detected}")

    if not detected:
        print("      Hakuna kitu kilichogundulika kwenye picha hii.")
        return None

    print("[2/4] Inatafuta targets za ChEMBL...")
    all_target_ids = set()
    for cls in detected:
        result = resolve_targets_for_class(cls)
        if "error" in result:
            print(f"      TAHADHARI: {result[\'error\']}")
            continue
        all_target_ids.update(result["target_chembl_ids"])
    print(f"      Targets: {sorted(all_target_ids)}")

    if not all_target_ids:
        print("      Hakuna target zilizopatikana.")
        return None

    print("[3/4] Inapakia efficacy na toxicity data...")
    efficacy_df = pd.read_csv(efficacy_csv)
    toxicity_df = fetch_toxicity_data(target_chembl_ids=list(all_target_ids))

    print("[4/4] Inapanga dawa...")
    ranked, status = rank_drug_candidates(
        efficacy_df, toxicity_df, list(all_target_ids),
        patient_allergy_smiles=patient_allergy_smiles,
    )

    if status != "OK":
        print(status)
        return None

    print("\n--- MATOKEO (Pendekezo la awali - SIYO agizo la matibabu) ---")
    print(ranked.to_string(index=False))
    return ranked

print("Pipeline tayari kutumika.")

### Mfano wa matumizi

```python
ranked_drugs = run_pipeline(
    image_path="picha_ya_mgonjwa.jpg",
    model_path="best.pt",                          # kutoka Sehemu 1 (training)
    efficacy_csv="offline_drug_training_data.csv",  # kutoka Sehemu 2
    patient_allergy_smiles=None,                    # ongeza SMILES za mzio wa mgonjwa hapa
)
```

Badilisha `image_path` na `model_path` kulingana na faili zako halisi kabla ya kuendesha.

In [ ]:
# ranked_drugs = run_pipeline(
#     image_path="picha_ya_mgonjwa.jpg",
#     model_path="best.pt",
#     efficacy_csv="offline_drug_training_data.csv",
# )

## Sehemu 6: Yanayohitajika Kabla ya Matumizi ya Kliniki

1. **Kamilisha `CLASS_TO_ORGANISM`** kwa `class_names` halisi za dataset lako la bakteria (`data.yaml`)
2. **Ongeza database ya wagonjwa** yenye historia ya mzio/matibabu (kwa sasa `patient_allergy_smiles` ni parameter tu)
3. **Thibitisha thresholds za Selectivity Index** (10, 2) na mtaalamu wa famasia/fasihi ya kisayansi
4. **Ongeza antibiogram/resistance data** halisi kwa bakteria (siyo efficacy dhidi ya target pekee)
5. **Fuata mchakato wa uthibitisho wa kikliniki na usajili** (TMDA/FDA/EMA n.k.) kabla ya matumizi na wagonjwa halisi
